# Fault Tolerance: RDD Lineage and Partition Recomputation

**Module**: 11_fault_tolerance_and_reliability  
**Notebook**: 01_lineage_and_recomputation.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: How Spark recovers lost partitions via deterministic RDD lineage graph re-evaluation without data replication.
- **Why does this topic exist?**: Lineage recomputation is Spark's core resilience mechanism.
- **What problem does it solve?**: Understanding what gets recomputed when worker nodes fail.
- **Where is it used in real-world Data Engineering?**: Spot instances and preemptible cloud clusters.

---

## 2. Lineage Recomputation Mechanics: If partition P2 fails, Spark traces P2's parent lineage back to source files and recomputes only P2.

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Lineage_Recompute_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
rdd = spark.sparkContext.parallelize(range(50), 4).map(lambda x: x*2).filter(lambda x: x > 20)
print("Lineage graph of transformations:")
print(rdd.toDebugString().decode('utf-8'))
print(f"Computed count: {rdd.count()}")

---

## Experiment: Lineage Recomputation Verification

- **1. What are we testing?**: Verify that lost partitions are reconstructed from upstream lineage.
- **2. What do we expect?**: Recomputation succeeds deterministically.

In [ ]:
print('Lineage graph has depth:', len(rdd.toDebugString().splitlines()))

**3. What happened?**: Lineage tree verified.  
**4. Why did it happen?**: Spark retains the complete sequence of deterministic operations.  
**5. Practical Engineering Lesson**: Deterministic transformations ensure exact partition reproduction upon failure.

---

## Practical Exercises

### Exercise 1 (Beginner): Inspect Lineage Depth
**Problem Statement**: Count transformation steps in toDebugString.  
**Input / Context**: rdd

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Non-Deterministic Hazard
**Problem Statement**: Why random numbers break lineage recomputation.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Lineage Truncation
**Problem Statement**: When to truncate lineage.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Inspect Lineage Depth):
```python
print(len(rdd.toDebugString().splitlines()))
```

#### Solution 2 (Non-Deterministic Hazard):
```python
print('Recomputed partitions will have different random values, corrupting joins.')
```

#### Solution 3 (Lineage Truncation):
```python
print('Use checkpoint() when lineage depth exceeds ~50 steps to prevent StackOverflowError.')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Lineage Resilience**: Recomputes lost partitions on the fly.
- **Zero Replication**: Saves network bandwidth compared to HDFS replication.

### Essential Technical Terms
- `Lineage`: Acyclic transformation graph.
- `Recomputation`: Evaluating parent steps for lost slices.

### Common Engineering Mistakes
- **Non-deterministic functions**: e.g. random.random() inside lineage causes corrupt recomputation.

### Interview Questions on This Topic
- **Q: How does Spark handle a lost partition?**
  - *A*: It recomputes only the lost partition from parent dependencies using its lineage graph.

### What to Study Next
Open 02_executor_failure.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')